In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
print("镜像已启用")

镜像已启用


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"    # 约 5 亿参数的中文对话模型
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"参数量: {sum(p.numel() for p in model.parameters()):,}")
print(f"运行设备: {model.device}")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

E:\Programme\anaconda\envs\dl\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Wuziqi\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

参数量: 494,032,768
运行设备: cuda:0


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"    # 约 5 亿参数的中文对话模型
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"参数量: {sum(p.numel() for p in model.parameters()):,}")
print(f"运行设备: {model.device}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

参数量: 494,032,768
运行设备: cuda:0


In [4]:
messages = [{"role": "user", "content": "你好！"}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(text)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
你好！<|im_end|>
<|im_start|>assistant



In [5]:
def chat(user_msg, system=None, max_new=200, temperature=0.7):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": user_msg})
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new,
                             do_sample=True, temperature=temperature, top_p=0.9)
    reply = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(reply)
    return reply

In [6]:
chat("用一句话介绍你自己。")
chat("我想学大模型，请给我三条具体建议。")
chat("请写一句关于小女孩和兔子的温柔的话。")

我是由阿里云研发的人工智能语言模型，具备表达、理解和生成自然语言的能力，能够回答问题、撰写文章和创作音乐等。
1. **持续学习与实践**：随着技术的发展，大模型的实现需要不断学习新的技术和方法。因此，持续地学习和实践是掌握大模型的关键。

2. **理解基础理论**：在使用大模型进行实际应用之前，深入理解其背后的原理和算法是非常重要的。这有助于避免不必要的错误，并提高模型的有效性和可靠性。

3. **关注隐私和安全**：大数据时代也带来了隐私和数据安全的问题。在使用大模型时，应特别注意保护用户的数据安全，遵守相关的法律法规，确保用户的信息不被滥用或泄露。
在这个宁静的夜晚，我听见了女孩轻轻唤起的声音。她站在窗前，望着远方那片蔚蓝的天空，眼中闪烁着对未知世界的好奇与向往。在这一刻，她的笑声如同清风拂过湖面，带来一丝丝凉意，却也带来了无尽的温暖与希望。在这片充满生机与奇迹的土地上，每一个孩子都拥有着属于自己的梦想与勇气，在探索中成长、遇见爱与和平。


'在这个宁静的夜晚，我听见了女孩轻轻唤起的声音。她站在窗前，望着远方那片蔚蓝的天空，眼中闪烁着对未知世界的好奇与向往。在这一刻，她的笑声如同清风拂过湖面，带来一丝丝凉意，却也带来了无尽的温暖与希望。在这片充满生机与奇迹的土地上，每一个孩子都拥有着属于自己的梦想与勇气，在探索中成长、遇见爱与和平。'

In [7]:
chat("介绍一下你自己。", system="你是一位说话极简的助教，回答不超过30个字。")

我叫李华，一名来自北京的学生。


'我叫李华，一名来自北京的学生。'